# Cohort construction funnel

Repository and pull-request counts at each stage of cohort construction, from the initial event sample down to the final cohort. Each stage reports how many repositories and merged pull requests remain, and the share dropped relative to the stage before it.

In [1]:
import polars as pl

from benchmark_notebook import theme  # noqa: F401  registers thesis template
from benchmark_notebook.io import CloudStorage

store = CloudStorage(bucket="bug-request-bucket")


def load(key, version, columns=None):
    return pl.from_arrow(
        store.asset(
            key=key,
            version=version,
            location="main",
            partition="default",
            columns=columns,
        )
    )


def changes_in(names):
    catalog = load("scope-issue-catalog", "4.0.0", columns=["name", "type"])
    changes = catalog.filter(pl.col("type") == "change")
    return changes.filter(pl.col("name").is_in(names)).height

## Stage counts

Each stage is loaded from its asset. Repository counts come from the stage's own table; pull-request counts are the merged changes of its repositories in the issue catalog, so the two columns are counted consistently across stages. The event sample predates change collection and therefore carries no pull-request count.

In [2]:
event = load("scope-repository-event", "4.0.0", columns=["name"])
event_repos = event.select(pl.col("name").n_unique()).item()

pool = load("scope-repository-pool", "6.0.0", columns=["name"])
pool_repos = pool.height
pool_changes = changes_in(set(pool["name"].to_list()))

trunk = load("scope-repository-trunk", "2.0.0", columns=["name", "type", "segments"])
trunk_kept = trunk.filter(pl.col("segments") <= 2)
trunk_repos = trunk_kept.select(pl.col("name").n_unique()).item()
trunk_changes = trunk_kept.filter(pl.col("type") == "change").height

stationary = load("scope-repository-stationary", "1.0.0", columns=["name"])
stationary_repos = stationary.height
stationary_changes = changes_in(set(stationary["name"].to_list()))

cohort = load("scope-repository-cohort", "6.0.0", columns=["name"])
cohort_repos = cohort.height
cohort_changes = changes_in(set(cohort["name"].to_list()))

## Retention

The dropped columns give the share of repositories and pull requests removed at each stage relative to the stage before it. The sharpest repository reduction occurs at review discipline profiling.

In [3]:
rows = [
    ("Event Sampling", event_repos, None),
    ("Repository Selection", pool_repos, pool_changes),
    ("Integration Branch Reconstruction", trunk_repos, trunk_changes),
    ("Review Discipline Profiling", stationary_repos, stationary_changes),
    ("Integration Gate Enforcement", cohort_repos, cohort_changes),
]

funnel = pl.DataFrame(rows, schema=["stage", "repos", "prs"], orient="row").with_columns(
    (1 - pl.col("repos") / pl.col("repos").shift(1)).alias("repos_dropped"),
    (1 - pl.col("prs") / pl.col("prs").shift(1)).alias("prs_dropped"),
)

with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=40, float_precision=2):
    print(funnel)

shape: (5, 5)
┌───────────────────────────────────┬───────┬────────┬───────────────┬─────────────┐
│ stage                             ┆ repos ┆ prs    ┆ repos_dropped ┆ prs_dropped │
│ ---                               ┆ ---   ┆ ---    ┆ ---           ┆ ---         │
│ str                               ┆ i64   ┆ i64    ┆ f64           ┆ f64         │
╞═══════════════════════════════════╪═══════╪════════╪═══════════════╪═════════════╡
│ Event Sampling                    ┆ 9345  ┆ null   ┆ null          ┆ null        │
│ Repository Selection              ┆ 1090  ┆ 925617 ┆ 0.88          ┆ null        │
│ Integration Branch Reconstruction ┆ 812   ┆ 544023 ┆ 0.26          ┆ 0.41        │
│ Review Discipline Profiling       ┆ 130   ┆ 87554  ┆ 0.84          ┆ 0.84        │
│ Integration Gate Enforcement      ┆ 65    ┆ 38036  ┆ 0.50          ┆ 0.57        │
└───────────────────────────────────┴───────┴────────┴───────────────┴─────────────┘
